# Buoyancy Production, Scalar Variance, and Mixing-Length — Rayleigh–Bénard Example

This notebook illustrates buoyancy production in the TKE budget and scalar-variance production
via gradient–diffusion closures, then applies a **mixing-length** model to a simple
Rayleigh–Bénard (RB) configuration (heated from below, cooled from above).

**Closures used** (Boussinesq):
- Scalar flux: $\overline{u_j'\theta'} = -\kappa_t\,\partial_j\overline{\theta}$, with $\kappa_t = \nu_t/\mathrm{Pr}_t$.
- Buoyancy production in TKE: $P_b = -\dfrac{\nu_t}{\mathrm{Pr}_t} N^2$, with $N^2 = g\alpha_T\,\partial_y\overline{\theta}$.
- Dissipation (local): $\varepsilon = C_\varepsilon\,\dfrac{k^{3/2}}{\ell_m}$.
- Eddy viscosity (mixing-length / k-based): $\nu_t = C_m\,\ell_m\sqrt{k}$, with $C_m\equiv C_\varepsilon^{1/3}$.
- Scalar-variance production: $\mathcal{P}_\vartheta = \kappa_t (\partial_y\overline{\theta})^2$.

Under **local equilibrium** (steady, 1-D, transport neglected): $P_b \approx \varepsilon$.
Combining gives a closed algebraic relation for $k$:
\begin{equation}
k(y) \;=\; \frac{C_m}{C_\varepsilon}\,\frac{\ell_m(y)^2}{\mathrm{Pr}_t}\,\big|N^2(y)\big|.
\end{equation}
This is convenient for buoyancy-dominated regions with weak mean shear.

## 1) Parameters and RB base state
We take a vertical domain $0\le y\le H$ with bottom temperature $T_b$ and top temperature $T_t$.
The conductive mean profile is $\overline{T}(y)=T_b - (\Delta T/H)\,y$ with $\Delta T=T_b-T_t>0$.
Then $\partial_y\overline{T}=-\Delta T/H$ and
$N^2=g\alpha_T\,\partial_y\overline{T}= -g\alpha_T\,\Delta T/H<0$ (unstable).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize']=(6,4)
plt.rcParams['figure.dpi']=200
plt.rcParams['savefig.dpi']=300
plt.rcParams['font.size']=12

# Physical / modeling parameters
g = 9.81            # m/s^2
alpha_T = 3.0e-3    # 1/K (illustrative)
nu = 1.0e-6         # m^2/s (kinematic viscosity)
kappa = 1.4e-7      # m^2/s (thermal diffusivity)
Pr = nu/kappa       # molecular Prandtl

H = 0.2             # m, fluid layer height
Tb = 310.0          # K
Tt = 300.0          # K
dT = Tb - Tt        # K (>0)

Ceps = 0.6          # C_epsilon
Cm   = Ceps**(1.0/3.0)
Pr_t = 0.9          # turbulent Prandtl
sig_k = 1.0         # gradient-diffusion for k (not used in local eq)

# Grid in y
Ny = 300
y  = np.linspace(0.0, H, Ny)

# Mean temperature profile and N^2
Tbar = Tb - dT * (y/H)
dTdy = - dT / H * np.ones_like(y)
N2   = g * alpha_T * dTdy  # Negative in RB (unstable)

# Rayleigh number
Ra = g * alpha_T * dT * H**3 / (nu * kappa)
print(f"Prandtl = {Pr:.2f},  Rayleigh = {Ra:.3e}")

## 2) Mixing length and closures
We choose a wall-aware mixing length that vanishes at plates, e.g.,
$\ell_m(y) = c_\ell\,H\,\sqrt{(y/H)(1-y/H)}$, with $c_\ell\in(0,1)$.
Then we compute $k(y)$ from the local-equilibrium relation, and derive
$\nu_t(y)=C_m\,\ell_m\sqrt{k}$, $\kappa_t=\nu_t/\mathrm{Pr}_t$, and
the budget terms $P_b(y)$, $\varepsilon(y)$, and $\mathcal{P}_\vartheta(y)$.

In [ ]:
c_ell = 0.25  # dimensionless
lm = c_ell * H * np.sqrt((y/H) * (1.0 - y/H))

# Local-equilibrium k from Pb = eps: k = (Cm/Ceps) * (lm^2/Pr_t) * |N2|
k_local = (Cm/Ceps) * (lm**2 / Pr_t) * np.abs(N2)

# Eddy viscosity and turbulent diffusivity
nu_t = Cm * lm * np.sqrt(np.maximum(k_local, 0.0))
kap_t = nu_t / Pr_t

# Budget terms
Pb = - nu_t * N2 / Pr_t                     # >0 for unstable (N2<0)
eps = Ceps * (k_local**1.5) / np.maximum(lm, 1e-12)  # avoid divide-by-zero at walls
Pth = kap_t * (dTdy**2)                     # scalar-variance production

fig, ax = plt.subplots()
ax.plot(y/H, lm, label='mixing length $\\ell_m$')
ax.set_xlabel('y/H'); ax.set_ylabel('Length [m]')
ax.set_title('Wall-aware Mixing Length')
ax.legend(frameon=False)
plt.show()

fig, ax = plt.subplots()
ax.plot(y/H, k_local)
ax.set_xlabel('y/H'); ax.set_ylabel('k [m$^2$/s$^2$]')
ax.set_title('Local-Equilibrium TKE from Buoyancy')
plt.show()

fig, ax = plt.subplots()
ax.plot(y/H, Pb, label='$P_b$')
ax.plot(y/H, eps, label='$\\varepsilon$')
ax.set_xlabel('y/H'); ax.set_ylabel('[m$^2$/s$^3$]')
ax.set_title('TKE Budget Terms: Buoyancy and Dissipation')
ax.legend(frameon=False)
plt.show()

fig, ax = plt.subplots()
ax.plot(y/H, Pth)
ax.set_xlabel('y/H'); ax.set_ylabel(r'$\mathcal{P}_\vartheta$ [K$^2$/s]')
ax.set_title('Scalar-Variance Production')
plt.show()


## 3) Notes on interpretation
- In unstable RB, $N^2<0$ so $P_b>0$ and drives turbulence.
- The local-equilibrium model gives $k\propto \ell_m^2 |N^2|$, so $k$ peaks away from plates where $\ell_m$ is largest.
- Including diffusion would smooth $k(y)$ further; this notebook focuses on the algebraic link for clarity.
- $\mathcal{P}_\vartheta$ is nonzero wherever $|\partial_y\overline{T}|>0$ and scales with $\kappa_t$.